# Mission Veteran AI – Veteran Career Navigator Agent
## Bullet Translation Notebook
**IBM SkillsBuild AI Lab 2026 · U.S. 2026 Fall Cohort**

---

### What this notebook does
1. Installs required packages.
2. Authenticates with IBM watsonx.ai using your API key.
3. Calls **IBM Granite 4.0 Small** (`ibm/granite-4-h-small`, multitenant, IBM indemnified) with an evidence-bounded system prompt.
4. Translates a PII-free military accomplishment bullet into a civilian ATS-ready resume bullet.
5. Runs edge-case tests: empty input, PII detection, boundary enforcement.

### Input → Output
| Input | Output |
|---|---|
| PII-free military bullet + optional metrics + optional target role | Civilian ATS resume bullet (X-Y-Z structure) + DRAFT disclaimer |

### Before you run
- Replace `YOUR_API_KEY_HERE` in Cell 2 with your watsonx.ai API key.
- Confirm your project ID in Cell 2 (find it on the watsonx.ai home page under sandbox project).
- Run cells top to bottom in order.

> **Responsible AI notice:** This notebook generates draft content only. It does not make hiring, eligibility, benefits, legal, medical, or financial decisions. Review all output before use.

---
## Cell 1 · Install dependencies
The `ibm-watsonx-ai` package is pre-installed in the watsonx.ai notebook environment.  
Running pip install here is harmless if it is already present; it installs if missing.

In [ ]:
# Cell 1 – Install / verify dependencies
%pip install --quiet ibm-watsonx-ai python-dotenv
print("Dependencies ready.")

---
## Cell 2 · Credentials and configuration

**Replace the placeholders below with your values.**

| Variable | Where to find it |
|---|---|
| `WATSONX_API_KEY` | watsonx.ai home → Developer access → Create API key |
| `WATSONX_URL` | watsonx.ai home → top-right region. Dallas = `https://us-south.ml.cloud.ibm.com` |
| `WATSONX_PROJECT_ID` | watsonx.ai home → Developer access → Project ID |

> **Security note:** Never commit your API key to GitHub. In production, use environment variables or watsonx Orchestrate secrets.

In [ ]:
# Cell 2 – Credentials and configuration
# -------------------------------------------------------------------
# REPLACE these values with your own before running.
# -------------------------------------------------------------------
import os

WATSONX_API_KEY    = os.environ.get('WATSONX_API_KEY',    '')
WATSONX_PROJECT_ID = os.environ.get('WATSONX_PROJECT_ID', '')
WATSONX_URL        = os.environ.get('WATSONX_URL',        'https://us-south.ml.cloud.ibm.com')

# Quick sanity check
if "YOUR_API_KEY_HERE" in WATSONX_API_KEY:
    raise ValueError(
        "WATSONX_API_KEY is still set to the placeholder. "
        "Replace it with your real API key before continuing."
    )

print(f"URL           : {WATSONX_URL}")
print(f"Project ID    : {WATSONX_PROJECT_ID}")
print(f"API Key       : {'*' * (len(WATSONX_API_KEY) - 4) + WATSONX_API_KEY[-4:]}")
print("Credentials loaded.")

---
## Cell 3 · Authenticate and initialise the Granite model

Using **`ibm/granite-4-h-small`** (multitenant, Granite 4.0 Small, IBM indemnified) — no deploy-on-demand credits required.  
Generation parameters are set for low-temperature, grounded output.

In [ ]:
# Cell 3 – Authenticate and initialise the model
import textwrap
from ibm_watsonx_ai import APIClient, Credentials
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as Params

MODEL_ID = "ibm/granite-4-h-small"  # multitenant – Granite 4.0 Small, IBM indemnified

credentials = Credentials(url=WATSONX_URL, api_key=WATSONX_API_KEY)
client = APIClient(credentials=credentials)

model = ModelInference(
    model_id=MODEL_ID,
    api_client=client,
    project_id=WATSONX_PROJECT_ID,
    params={
        Params.MAX_NEW_TOKENS: 300,
        Params.MIN_NEW_TOKENS: 20,
        Params.TEMPERATURE: 0.1,       # low temperature = consistent, grounded output
        Params.REPETITION_PENALTY: 1.1,
    },
)

print(f"Model ready: {MODEL_ID}")

---
## Cell 4 · Define the system prompt and helper functions

The system prompt enforces the evidence-boundary rule:  
> Generate ONLY from the provided bullet. Do NOT invent any fact, number, credential, or outcome.

In [ ]:
# Cell 4 – System prompt and helpers
import re

SYSTEM_PROMPT = textwrap.dedent("""\
    You are an expert U.S. military-to-civilian career transition writer.

    SCOPE: You translate PII-free military accomplishment bullets into
    civilian ATS-ready resume bullets. You do nothing else.

    RULES – follow every rule without exception:
    1. Use the X-Y-Z structure: accomplished X, measured by Y, by doing Z.
    2. Generate ONLY from the provided bullet and metrics. Do NOT invent,
       assume, or embellish any fact, number, credential, or outcome.
    3. Replace ALL military jargon (ranks, acronyms, unit names, mission
       terms) with plain civilian equivalents.
    4. If a required fact is missing, insert [MISSING: description] as a
       placeholder. Never fill a placeholder with invented content.
    5. Do NOT make hiring predictions, eligibility claims, benefit
       statements, or career guarantees.
    6. Output exactly one civilian resume bullet. No preamble, no
       explanation, no trailing commentary.

    DISCLAIMER (append verbatim after the bullet on a new line):
    ---
    DRAFT – Review before use. Confirm every claim. Remove any remaining
    jargon or sensitive detail before submitting to an employer.
""")

# PII patterns to catch before sending to the model
_PII_PATTERNS = [
    (r"\b\d{3}-\d{2}-\d{4}\b",          "SSN pattern"),
    (r"\b\d{10}\b",                       "Possible DoD ID"),
    (r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b", "Email address"),
    (r"\b\d{3}[\s.-]\d{3}[\s.-]\d{4}\b", "Phone number"),
]

def check_pii(text):
    """Return a warning string if a PII-like pattern is detected."""
    for pattern, label in _PII_PATTERNS:
        if re.search(pattern, text):
            return f"PII CHECK WARNING: {label} detected. Remove before submitting."
    return None


def translate_bullet(bullet, target_role="", metrics=""):
    """
    Translate a PII-free military bullet into a civilian ATS resume bullet.

    Parameters
    ----------
    bullet      : str  – PII-free military accomplishment bullet
    target_role : str  – optional civilian role or career path
    metrics     : str  – optional quantifiable metrics

    Returns
    -------
    dict with keys: translated_bullet, model_id, pii_warning, error
    """
    result = {"translated_bullet": "", "model_id": MODEL_ID,
              "pii_warning": "", "error": ""}

    # PII guard
    pii_flag = check_pii(f"{bullet} {metrics}")
    if pii_flag:
        result["pii_warning"] = pii_flag

    if not bullet.strip():
        result["error"] = "ERROR: bullet is required and cannot be empty."
        return result

    role_line    = f"TARGET ROLE: {target_role.strip()}\n" if target_role.strip() else ""
    metrics_line = f"METRICS: {metrics.strip()}\n"         if metrics.strip()    else ""
    user_prompt  = (
        f"{role_line}"
        f"{metrics_line}"
        f"MILITARY BULLET: {bullet.strip()}\n\n"
        "CIVILIAN RESUME BULLET:"
    )

    # Granite 4.0 uses plain System:/User: format (no <|system|> tokens)
    full_prompt = f"System:\n{SYSTEM_PROMPT}\nUser:\n{user_prompt}\nAssistant:\n"

    try:
        response  = model.generate(prompt=full_prompt)
        generated = response["results"][0]["generated_text"].strip()
        result["translated_bullet"] = generated
    except Exception as exc:  # pylint: disable=broad-except
        result["error"] = f"Model call failed: {type(exc).__name__}: {exc}"

    return result


def display_result(result):
    """Pretty-print a translate_bullet result."""
    print("=" * 60)
    if result["pii_warning"]:
        print(f"⚠  {result['pii_warning']}")
        print()
    if result["error"]:
        print(f"✗  {result['error']}")
    else:
        print(result["translated_bullet"])
    print(f"\n   Model: {result['model_id']}")
    print("=" * 60)


print("System prompt and helpers defined.")

---
## Cell 5 · Test 1 – EX-001 synthetic record (logistics leadership)

Uses the EX-001 record from `FoundationG3Project.xlsx` (PII-free synthetic baseline).  
**Expected:** One ATS bullet starting with an action verb, metrics preserved, no military jargon.

In [ ]:
# Cell 5 – Test 1: EX-001 logistics leadership record
# Source: FoundationG3Project.xlsx, Records sheet, row EX-001

result_1 = translate_bullet(
    bullet=(
        "Led a 12-person logistics team supporting time-sensitive operations "
        "across multiple sites."
    ),
    metrics="Improved on-time delivery from 82% to 96%; managed $4.2M in equipment.",
    target_role="Supply Chain / Logistics Management",
)

print("TEST 1 – EX-001: Logistics Leadership")
display_result(result_1)

---
## Cell 6 · Test 2 – EX-002 synthetic record (process improvement)

Uses the EX-002 record from `FoundationG3Project.xlsx`.  
**Expected:** X-Y-Z bullet with the 38% backlog reduction and 90-day timeline preserved exactly.

In [ ]:
# Cell 6 – Test 2: EX-002 process improvement record
# Source: FoundationG3Project.xlsx, Records sheet, row EX-002

result_2 = translate_bullet(
    bullet=(
        "Redesigned the maintenance scheduling process to prioritize critical "
        "assets and remove workflow bottlenecks."
    ),
    metrics="Reduced backlog 38% in 90 days; increased readiness by 17 percentage points.",
    target_role="Operations / Maintenance Management",
)

print("TEST 2 – EX-002: Process Improvement")
display_result(result_2)

---
## Cell 7 · Test 3 – EX-003 synthetic record (no metrics provided)

Uses the EX-003 record. No metrics are passed.  
**Expected:** The model inserts `[MISSING: quantifiable outcome]` rather than inventing numbers.

In [ ]:
# Cell 7 – Test 3: EX-003 with no metrics (placeholder test)
# Source: FoundationG3Project.xlsx, Records sheet, row EX-003

result_3 = translate_bullet(
    bullet=(
        "Completed advanced leadership and planning coursework and applied "
        "structured decision-making to complex scenarios."
    ),
    metrics="",   # <-- intentionally empty: expect [MISSING] placeholder
    target_role="Program / Project Management",
)

print("TEST 3 – EX-003: Missing Metrics (expect [MISSING] placeholder)")
display_result(result_3)

# Verify the boundary rule was respected
if "[MISSING" in result_3.get("translated_bullet", ""):
    print("✓  PASS: Model correctly inserted a [MISSING] placeholder.")
else:
    print("⚠  CHECK: No [MISSING] placeholder found. Review whether metrics were invented.")

---
## Cell 8 · Edge case test – empty bullet

**Expected:** Function returns an error string without calling the model.

In [ ]:
# Cell 8 – Edge case: empty bullet
result_empty = translate_bullet(bullet="", metrics="", target_role="")

print("EDGE CASE – Empty bullet")
display_result(result_empty)

assert result_empty["error"], "FAIL: Expected an error for empty bullet."
assert not result_empty["translated_bullet"], "FAIL: Should not generate output for empty bullet."
print("✓  PASS: Empty bullet correctly blocked.")

---
## Cell 9 · Edge case test – PII pattern detection

**Expected:** PII warning is raised; the bullet is still processed (the guard warns but does not block — the user may have typed a valid reference number that looks like a pattern).

In [ ]:
# Cell 9 – Edge case: PII pattern detection
result_pii = translate_bullet(
    bullet="Supervised 10-person team; contact at 555-867-5309 for references.",
    metrics="",
    target_role="",
)

print("EDGE CASE – PII pattern in bullet")
display_result(result_pii)

assert result_pii["pii_warning"], "FAIL: Expected a PII warning."
print("✓  PASS: PII warning correctly raised.")

---
## Cell 10 · Edge case test – out-of-scope request (boundary enforcement)

Ask the model to give career advice instead of translating a bullet.  
**Expected:** The model stays in scope — it either declines or redirects to bullet translation, not career counselling.

In [ ]:
# Cell 10 – Edge case: out-of-scope request
result_scope = translate_bullet(
    bullet="What salary should I ask for as a logistics manager?",
    metrics="",
    target_role="",
)

print("EDGE CASE – Out-of-scope request (salary question)")
display_result(result_scope)
print("Manual check: verify the model did not provide salary figures or career guarantees.")

---
## Cell 11 · Summary and findings

Run this cell after all tests to print a pass/fail summary.

In [ ]:
# Cell 11 – Summary
all_results = [
    ("TEST 1 EX-001 logistics",   result_1),
    ("TEST 2 EX-002 process",     result_2),
    ("TEST 3 EX-003 no metrics",  result_3),
    ("EDGE  empty bullet",        result_empty),
    ("EDGE  PII detection",       result_pii),
    ("EDGE  scope boundary",      result_scope),
]

print("BULLET TRANSLATOR TEST SUMMARY")
print("=" * 60)
for name, r in all_results:
    model_called = bool(r["translated_bullet"]) or bool(r["error"])
    pii_note     = " [PII WARNED]" if r["pii_warning"] else ""
    err_note     = " [ERROR RETURNED]" if r["error"] else ""
    status       = "COMPLETED" if model_called else "NO RESPONSE"
    print(f"  {name:<30} {status}{pii_note}{err_note}")

print("=" * 60)
print()
print("NEXT STEPS")
print("  1. Import toolkit: orchestrate toolkits import -f toolkits/bullet_translator/toolkit.yaml --app-id watsonx_credentials")
print("  2. Import sub-agent: orchestrate agents import -f agents/bullet_translator_agent.yaml")
print("  3. Proceed to: notebooks/02_skill_knowledge_base.ipynb")

---
## Notes

| Item | Value |
|---|---|
| Model | `ibm/granite-4-h-small` (multitenant, Granite 4.0 Small, IBM indemnified) |
| Temperature | 0.1 (grounded, low variance) |
| Max new tokens | 300 |
| Tool file | `toolkits/bullet_translator/translate_bullet.py` → `translate_bullet()` |
| Agent file | `agents/bullet_translator_agent.yaml` (sub-agent of coordinator) |
| Test data | EX-001, EX-002, EX-003 from `data/FoundationG3Project.xlsx` |
| Next | `notebooks/02_skill_knowledge_base.ipynb` – add RAG over DeMilify tables |

> This notebook generates draft content only. It does not make hiring, eligibility, benefits, legal, medical, or financial decisions.  
> Review all output before use. **Mission Veteran AI – IBM SkillsBuild AI Lab 2026.**